In [ ]:
# ======================
# CONFIGURATION DU CHEMIN DU PROJET
# ======================
import sys
import os

# Remonter à la racine du projet (depuis notebooks/training/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Racine du projet ajoutée au PYTHONPATH : {project_root}")
print(f"   sys.path : {sys.path[0]}")

✅ Racine du projet ajoutée au PYTHONPATH : c:\Users\SOL\Downloads\sentiment_analysis_project
   sys.path : c:\Users\SOL\Downloads\sentiment_analysis_project


## importation de model et base de donnée

In [ ]:
from src.preprocessing import preprocess_full_dataset
from src.dataset import create_dataloaders
from src.models import BiLSTMAttention

# Exécuter le prétraitement
df_clean, recommended_length = preprocess_full_dataset(r'C:\Users\SOL\Downloads\sentiment_analysis_project\data\sentiment_dataset.csv')
# Créer les dataloaders PyTorch
train_loader, test_loader, vocab, train_df, test_df = create_dataloaders(
    df_clean,
    max_length=60,
    batch_size=64
)

# Initialisation du modèle
model = BiLSTMAttention(
    vocab_size=len(vocab),
    embedding_dim=256,
    hidden_dim=128,
    num_layers=2,
    output_dim=3,
    dropout=0.5
)

print(f"🧠 Modèle BiLSTM + Attention initialisé :")
print(f"   - Type          : Bidirectionnel (forward + backward)")
print(f"   - Embedding dim : 256")
print(f"   - Hidden dim    : 128 (×2 = 256 avec bidirection)")
print(f"   - Couches       : 2")
print(f"   - Paramètres    : {sum(p.numel() for p in model.parameters()):,}")


🚀 PRÉTRAITEMENT DU DATASET

✅ Dataset chargé : 31,232 échantillons
🧹 Prétraitement en cours...
✅ 31,202 échantillons conservés

📏 Longueur recommandée (95e percentile) : 56 tokens

📚 Vocabulaire construit : 11703 mots uniques

📦 Dataloaders créés :
   - Train : 391 batches
   - Test  : 98 batches
🧠 Modèle BiLSTM + Attention initialisé :
   - Type          : Bidirectionnel (forward + backward)
   - Embedding dim : 256
   - Hidden dim    : 128 (×2 = 256 avec bidirection)
   - Couches       : 2
   - Paramètres    : 3,820,292


## l'entrainement

In [ ]:
from src.trainer import train_model
import torch
import torch.nn as nn

# Sélection du device (GPU si disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Entraînement avec monitoring complet
history, model_path = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=30,
    patience=4,
    save_dir=r"C:\Users\SOL\Downloads\sentiment_analysis_project\models\trainary",
    model_name="bilstm_attention",
    verbose=True,
)



🚀 DÉBUT DE L'ENTRAÎNEMENT : bilstm_attention
Epoch  | Train Loss   Train F1   Val Loss     Val F1     Val Acc    | Time   | Status      
--------------------------------------------------------------------------------
     1 | 0.9629       0.5117     0.8244       0.6224     0.6251     | 5     s | ★ BEST      
     2 | 0.8420       0.6122     0.7778       0.6568     0.6589     | 5     s | ★ BEST      
     3 | 0.7849       0.6524     0.7459       0.6783     0.6779     | 5     s | ★ BEST      
     4 | 0.7377       0.6796     0.7315       0.6952     0.6959     | 5     s | ★ BEST      
     5 | 0.6972       0.7016     0.7229       0.7005     0.6999     | 5     s | ★ BEST      
     6 | 0.6657       0.7168     0.7509       0.6909     0.6924     | 5     s | →           
     7 | 0.6399       0.7271     0.7192       0.7005     0.7005     | 5     s | →           
     8 | 0.6182       0.7397     0.7341       0.7064     0.7057     | 5     s | ★ BEST      
     9 | 0.6006       0.7466     0.74

## évaluation

In [ ]:
# Évaluation finale avec visualisations publication
import torch
from src.evaluate import evaluate_model
# Sélection du device (GPU si disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

results = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=['Négatif', 'Neutre', 'Positif'],
    save_dir="reports/figures",
    model_name="bilstm_attention"
)


📊 ÉVALUATION DU MODÈLE : bilstm_attention

Accuracy      : 0.7052
F1 Macro      : 0.7062
F1 Weighted   : 0.7049
ROC AUC Micro : 0.8692
ROC AUC Macro : 0.8642

Classification Report:
              precision    recall  f1-score   support

     Négatif     0.7102    0.6720    0.6906      1820
      Neutre     0.6468    0.6571    0.6519      2327
     Positif     0.7652    0.7875    0.7762      2094

    accuracy                         0.7052      6241
   macro avg     0.7074    0.7055    0.7062      6241
weighted avg     0.7050    0.7052    0.7049      6241

   → Matrice de confusion sauvegardée : confusion_matrix.png/csv
   → Courbes ROC sauvegardées : roc_curves.png

✅ Évaluation terminée - Résultats sauvegardés dans : reports\figures\bilstm_attention
